# 3D-SG Tapered Shell Segment: Timoshenko 6x6 vs 2D Solid

A wind-turbine blade is **tapered and twisted**, so a spanwise segment is an *aperiodic* 3-D
structure genome (SG). This tutorial runs the **MITC-RM tapered-segment** pipeline on three
BAR-URC segments and benchmarks every step against the FEniCS **3-D solid** segment solution
at the *same origin*:

1. **topological boundary extraction** &mdash; a mesh edge used by exactly one quad is a *free*
   edge; the two connected components of the free-edge graph are the end cross-sections. Each
   boundary line element inherits **layup and orientation from its parent quad** (verified
   identical to the dolfinx `create_submesh` facet-to-cell mapping);
2. **boundary rings** solved as 1-D RM/MITC cross-section SGs (Timoshenko $6\times6$ + warping
   $V_0, V_1$);
3. **tapered segment** &mdash; 2-D MITC4 RM surface elements over the full segment, boundary
   warping applied as Dirichlet data (no rigid-body constraints), Euler-Bernoulli $V_0$ then
   Timoshenko $V_1$ condensation to the $6\times6$ per unit length.

**Conventions used for these meshes** (BAR-URC `Shell_3D_Taper`):
- element-set labels are **0-indexed** (auto-detected by the reader);
- mesh nodes are the **laminate reference surface** (stack bottom) &rightarrow; `center_ref=False`
  (a mid-surface mesh would use `center_ref=True`);
- `shear='mitc_both'` is the default RM scheme; the **curvature argument** `k22` selects the
  initial hoop curvature: `'general'` (per-element geometric, flat elements *exactly* zero),
  `None` (uniform $-1/R$) or `'zero'`. The curvature terms are bounded: they touch only
  GA$_2$/GA$_3$/GJ and move them by $\lesssim 3\%$ (EA/EI are insensitive).

Shell segments 5 / 12 / 15 span the same stations as solid segments 4 / 11 / 14
(origins 18.97 / 43.10 / 53.45 m of the 100 m blade); the solid references were computed with
`opensg.core.solid.compute_stiffness` (boundary `Taper=False`, segment `Taper=True`) and are
bundled in `examples/data/benchmark/bar_urc_taper_solid_refs.npz`.

In [1]:
import os, sys, tempfile
import numpy as np

def _find_repo_root(d=None):
    d = os.path.abspath(d or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(d, "examples", "data")) and os.path.isfile(os.path.join(d, "pyproject.toml")):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            raise RuntimeError("OpenSG-TW repo root not found - run this notebook from inside the cloned repo")
        d = parent

CC = _find_repo_root()
for p in (CC, os.path.join(CC, "mitc_rm_segment")):
    if p not in sys.path:
        sys.path.insert(0, p)

from boundary_from_yaml import extract
from solve_segment_jax import solve_boundary_bundle
from compute_timo_taper import compute_timo_taper

D3   = os.path.join(CC, "examples", "data", "3d_yaml")
REFS = np.load(os.path.join(CC, "examples", "data", "benchmark", "bar_urc_taper_solid_refs.npz"))
TMP  = tempfile.mkdtemp(prefix="taper3dsg_")

# BAR-URC Shell_3D_Taper: mesh = laminate reference surface -> bottom-ref ABD
CFG = dict(center_ref=False)          # shear defaults to 'mitc_both'
LBL = ["ext", "sh2", "sh3", "tor", "bd2", "bd3"]

def show_compare(name, S, So):
    """Full 6x6 comparison: shell vs solid, %err on every non-zero C_ij."""
    S = 0.5 * (S + S.T); So = 0.5 * (So + So.T)
    thr = 1e-2 * max(np.abs(np.diag(So)).max(), np.abs(np.diag(S)).max())
    print("===== %s : non-zero terms (>1%% of max diagonal) =====" % name)
    print("  %-9s %12s %12s %9s" % ("term", "shell", "solid", "%err"))
    for i in range(6):
        for j in range(i, 6):
            if abs(So[i, j]) > thr or abs(S[i, j]) > thr:
                err = 100.0 * (S[i, j] - So[i, j]) / So[i, j]
                tag = ("%s-%s" % (LBL[i], LBL[j])) if i != j else LBL[i].upper()
                print("  %-9s %12.4e %12.4e %+8.1f%%" % (tag, S[i, j], So[i, j], err))

def run_segment(shell_id, solid_id):
    yml = os.path.join(D3, "BAR_URC_numEl_52_segment_%d.yaml" % shell_id)
    npz = os.path.join(TMP, "seg%d.npz" % shell_id)
    extract(yml, npz)                                    # free-edge boundary extraction
    b = np.load(npz, allow_pickle=True)
    rL = solve_boundary_bundle(b, "L", k22="general", **CFG)   # 1-D RM ring, geometric curvature
    rR = solve_boundary_bundle(b, "R", k22="general", **CFG)
    rT = compute_timo_taper(b, k22_mode="general", return_timo=True, **CFG)
    print()
    show_compare("seg %d LEFT boundary vs solid seg %d L" % (shell_id, solid_id),
                 np.asarray(rL["C6"]), REFS["seg%d_L" % solid_id])
    show_compare("seg %d RIGHT boundary vs solid seg %d R" % (shell_id, solid_id),
                 np.asarray(rR["C6"]), REFS["seg%d_R" % solid_id])
    show_compare("seg %d TAPERED SEGMENT vs solid seg %d" % (shell_id, solid_id),
                 np.asarray(rT["C6"]), REFS["seg%d_seg" % solid_id])
    return b, rL, rR, rT

## Example 1: segment 5 (span 17.24 - 20.69 m)

The two extracted boundary rings, coloured by layup (the carbon spar caps in bold): the
cross-section shrinks and twists from the left to the right station, and the spar-cap arc is
bounded by the shear webs at both ends.

In [2]:
b5, rL5, rR5, rT5 = run_segment(5, 4)

segment frame: |e1|=1.0000 |e2|=1.0000 |e3|=1.0000  max|off-diag|=6.2e-16  RH(e1xe2.e3) in [1.000,1.000]  min(e1.x)=-0.0177  mean(e3.-r)=0.0314  -> CHECK


free edges 128 -> 2 end cross-section(s)
beam axis = z ; 62/62 nodes on L/R cross-section
  L cross-section: 62 nodes, 64 edges, 4 junctions(deg>2)
  L-ring frame: |e1|=1.0000 |e2|=1.0000 |e3|=1.0000  max|off-diag|=3.9e-16  RH(e1xe2.e3) in [1.000,1.000]  min(e1.x)=-0.0140  mean(e3.-r)=0.0371  -> CHECK
  R cross-section: 62 nodes, 64 edges, 4 junctions(deg>2)
  R-ring frame: |e1|=1.0000 |e2|=1.0000 |e3|=1.0000  max|off-diag|=4.7e-16  RH(e1xe2.e3) in [1.000,1.000]  min(e1.x)=-0.0173  mean(e3.-r)=0.0236  -> CHECK


wrote C:\Users\bagla0\AppData\Local\Temp\taper3dsg_1lljwo32\seg5.npz



===== seg 5 LEFT boundary vs solid seg 4 L : non-zero terms (>1% of max diagonal) =====
  term             shell        solid      %err
  EXT         1.4827e+10   1.4302e+10     +3.7%
  ext-bd2    -7.2788e+08  -5.7883e+08    +25.7%
  ext-bd3    -4.6416e+09  -4.3425e+09     +6.9%
  SH2         1.4606e+09   1.4878e+09     -1.8%
  sh2-sh3    -2.2713e+08  -2.3120e+08     -1.8%
  SH3         4.7618e+08   5.1038e+08     -6.7%
  sh3-tor     3.9964e+08   4.3783e+08     -8.7%
  TOR         1.9045e+09   2.0228e+09     -5.8%
  BD2         6.9854e+09   6.6843e+09     +4.5%
  bd2-bd3     3.0423e+09   2.8814e+09     +5.6%
  BD3         1.8518e+10   1.7939e+10     +3.2%
===== seg 5 RIGHT boundary vs solid seg 4 R : non-zero terms (>1% of max diagonal) =====
  term             shell        solid      %err
  EXT         1.4695e+10   1.3582e+10     +8.2%
  ext-bd2    -4.5488e+08  -9.8003e+08    -53.6%
  ext-bd3    -4.0124e+09  -4.2777e+09     -6.2%
  SH2         1.4731e+09   1.1228e+09    +31.2%
  sh2-

In [3]:
import json
import matplotlib.pyplot as plt

sections = json.loads(str(b5["sections"]))
A11p = {si: sum(float(p[1]) for p in s["layup"]) for si, s in enumerate(sections)}
ax_i = int(b5["axis"]); cross = [j for j in range(3) if j != ax_i]
cmap = plt.get_cmap("tab10")
fig, axs = plt.subplots(1, 2, figsize=(12, 5))
for k, side in enumerate(("L", "R")):
    X = np.asarray(b5["%s_x" % side])[:, cross]
    cells = np.asarray(b5["%s_cells" % side]); sd = np.asarray(b5["%s_subdom" % side])
    for i, (a, c) in enumerate(cells):
        s = int(sd[i])
        carbon = any("carbon" in p[0] for p in sections[s]["layup"])
        axs[k].plot([X[a, 0], X[c, 0]], [X[a, 1], X[c, 1]], "-",
                    color=cmap(s % 10), lw=5 if carbon else 1.6, solid_capstyle="round")
    axs[k].set_title("%s ring (station %.2f m)" % (side, float(np.asarray(b5["%s_x" % side])[:, ax_i].mean())))
    axs[k].set_aspect("equal"); axs[k].grid(alpha=0.3)
    axs[k].set_xlabel("$x_2$ [m]"); axs[k].set_ylabel("$x_3$ [m]")
plt.tight_layout(); plt.show()

C:\Users\bagla0\AppData\Local\Temp\ipykernel_34948\1096421944.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Example 2: segment 12 (span 41.38 - 44.83 m)

In [4]:
_ = run_segment(12, 11)

segment frame: |e1|=1.0000 |e2|=1.0000 |e3|=1.0000  max|off-diag|=5.6e-16  RH(e1xe2.e3) in [1.000,1.000]  min(e1.x)=-0.0439  mean(e3.-r)=0.0168  -> CHECK


free edges 128 -> 2 end cross-section(s)
beam axis = z ; 62/62 nodes on L/R cross-section
  L cross-section: 62 nodes, 64 edges, 4 junctions(deg>2)
  L-ring frame: |e1|=1.0000 |e2|=1.0000 |e3|=1.0000  max|off-diag|=3.4e-16  RH(e1xe2.e3) in [1.000,1.000]  min(e1.x)=-0.0387  mean(e3.-r)=0.0151  -> CHECK
  R cross-section: 62 nodes, 64 edges, 4 junctions(deg>2)
  R-ring frame: |e1|=1.0000 |e2|=1.0000 |e3|=1.0000  max|off-diag|=2.0e-16  RH(e1xe2.e3) in [1.000,1.000]  min(e1.x)=-0.0439  mean(e3.-r)=0.0159  -> CHECK


wrote C:\Users\bagla0\AppData\Local\Temp\taper3dsg_1lljwo32\seg12.npz



===== seg 12 LEFT boundary vs solid seg 11 L : non-zero terms (>1% of max diagonal) =====
  term             shell        solid      %err
  EXT         1.4988e+10   1.4743e+10     +1.7%
  ext-bd2     2.1307e+08   2.0216e+08     +5.4%
  ext-bd3    -4.3424e+09  -4.6031e+09     -5.7%
  SH2         6.1591e+08   6.2561e+08     -1.6%
  TOR         3.0547e+08   3.2330e+08     -5.5%
  BD2         2.9313e+09   2.9105e+09     +0.7%
  bd2-bd3     6.0430e+08   5.3298e+08    +13.4%
  BD3         9.5865e+09   8.7509e+09     +9.5%
===== seg 12 RIGHT boundary vs solid seg 11 R : non-zero terms (>1% of max diagonal) =====
  term             shell        solid      %err
  EXT         1.4838e+10   1.4315e+10     +3.7%
  ext-bd2     2.5945e+08   2.3044e+08    +12.6%
  ext-bd3    -4.0632e+09  -4.3710e+09     -7.0%
  SH2         5.9379e+08   5.4411e+08     +9.1%
  TOR         2.5025e+08   2.4251e+08     +3.2%
  BD2         2.4778e+09   2.4431e+09     +1.4%
  bd2-bd3     4.4999e+08   3.5992e+08    +25.0%
  

## Example 3: segment 15 (span 51.72 - 55.17 m)

In [5]:
_ = run_segment(15, 14)

segment frame: |e1|=1.0000 |e2|=1.0000 |e3|=1.0000  max|off-diag|=6.6e-16  RH(e1xe2.e3) in [1.000,1.000]  min(e1.x)=-0.0448  mean(e3.-r)=0.0140  -> CHECK


free edges 128 -> 2 end cross-section(s)
beam axis = z ; 62/62 nodes on L/R cross-section
  L cross-section: 62 nodes, 64 edges, 4 junctions(deg>2)
  L-ring frame: |e1|=1.0000 |e2|=1.0000 |e3|=1.0000  max|off-diag|=5.3e-16  RH(e1xe2.e3) in [1.000,1.000]  min(e1.x)=-0.0448  mean(e3.-r)=0.0142  -> CHECK
  R cross-section: 62 nodes, 64 edges, 4 junctions(deg>2)
  R-ring frame: |e1|=1.0000 |e2|=1.0000 |e3|=1.0000  max|off-diag|=3.5e-16  RH(e1xe2.e3) in [1.000,1.000]  min(e1.x)=-0.0408  mean(e3.-r)=0.0137  -> CHECK


wrote C:\Users\bagla0\AppData\Local\Temp\taper3dsg_1lljwo32\seg15.npz



===== seg 15 LEFT boundary vs solid seg 14 L : non-zero terms (>1% of max diagonal) =====
  term             shell        solid      %err
  EXT         1.3326e+10   1.3031e+10     +2.3%
  ext-bd3    -3.2412e+09  -3.5277e+09     -8.1%
  SH2         4.4066e+08   4.5528e+08     -3.2%
  TOR         1.4593e+08   1.5426e+08     -5.4%
  BD2         1.6711e+09   1.6569e+09     +0.9%
  bd2-bd3     2.9777e+08   2.5335e+08    +17.5%
  BD3         5.8092e+09   5.0609e+09    +14.8%
===== seg 15 RIGHT boundary vs solid seg 14 R : non-zero terms (>1% of max diagonal) =====
  term             shell        solid      %err
  EXT         1.3201e+10   1.1970e+10    +10.3%
  ext-bd2     7.3430e+07   2.8321e+08    -74.1%
  ext-bd3    -2.9699e+09  -2.9999e+09     -1.0%
  SH2         4.3196e+08   3.8330e+08    +12.7%
  BD2         1.3942e+09   1.3082e+09     +6.6%
  bd2-bd3     2.0873e+08   8.4211e+07   +147.9%
  BD3         5.1082e+09   4.1209e+09    +24.0%
===== seg 15 TAPERED SEGMENT vs solid seg 14 : non

## Writer-independence check: JAX vs OpenSG boundary YAML

The same physical cross-section (segment 5's right end, station 20.69 m) written by **two
different tools** &mdash; our topological extractor and the official OpenSG
`ShellSegmentMesh._create_1Dyaml` &mdash; differs only in node/element *ordering* and
orientation-frame convention. `solve_1dyaml` canonicalizes both (edge direction CCW about the
section centroid from coordinates alone; symmetric direction-covariant discrete curvature), so
the two files must give the **same Timoshenko $6\times6$**:

In [6]:
from solve_1dyaml import solve as solve_1d

D1 = os.path.join(CC, "examples", "data", "1d_yaml")
A = solve_1d(os.path.join(D1, "bar_urc_seg5R_jax1d.yaml"), center_ref=False, verbose=True)
B = solve_1d(os.path.join(D1, "bar_urc_seg5R_opensg1d.yaml"), center_ref=False, verbose=True)
print("diag A x1e9:", np.array2string(np.diag(A) / 1e9, precision=4))
print("diag B x1e9:", np.array2string(np.diag(B) / 1e9, precision=4))
reldiff = np.max(np.abs(A - B)) / np.max(np.abs(A))
print("max |A-B| / max|A| = %.2e   (identical mesh content up to reordering)" % reldiff)
assert reldiff < 1e-5

  bar_urc_seg5R_jax1d.yaml: 62 nodes, 64 edges (2 flipped), 9 sections, k22[geom] range [-16.750, 16.478]


  bar_urc_seg5R_opensg1d.yaml: 62 nodes, 64 edges (2 flipped), 9 sections, k22[geom] range [-16.750, 16.478]
diag A x1e9: [14.6948  1.4729  0.3949  1.4407  5.183  18.1201]
diag B x1e9: [14.6948  1.4729  0.3949  1.4407  5.183  18.1201]
max |A-B| / max|A| = 4.27e-07   (identical mesh content up to reordering)


## Summary

- The **tapered-segment $6\times6$ tracks its own end cross-sections**: EA, EI$_2$, GJ and the
  shear terms land within $\sim$10% of the 3-D solid wherever the boundary cross-sections do.
- The remaining **EI$_3$ (edgewise) gap grows outboard and is *data-level*** &mdash; the official
  OpenSG FEniCS shell gives the same +10-15% vs solid at these stations (BAR-URC flatback
  trailing-edge representation), so it is a property of the shell cross-section model, not of
  the tapered-segment machinery.
- The **right end of each segment is stiffer than the same station seen from the next
  segment's left end**: each segment file freezes its layup (piecewise-constant SG), so the
  outboard face carries the inboard-frozen thicknesses. This is the data convention (the
  official OpenSG per-station pipeline uses **left** boundaries for exactly this reason).
- **Curvature terms are bounded**: only GA$_2$/GA$_3$/GJ respond ($\lesssim 3\%$ between
  `'general'`, uniform and `'zero'`), flat elements (webs) carry exactly zero, and even
  doubling every curvature moves no term more than $\sim$5%.